In [25]:
import numpy as np
from collections import Counter

In [26]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        """
        Initializes a Node in the Decision Tree.
        
        Args:
            feature (int, optional): The index of the feature to evaluate.
            threshold (float, optional): The threshold for the split.
            left (Node, optional): The left child Node (data <= threshold).
            right (Node, optional): The right child Node.
            value (int, optional): The predicted class label. Relevent only if this is a leaf.
        """
        # Attributes for Internal Nodes (Decision nodes)
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        
        # Attribute for Leaf Nodes (Terminal nodes)
        # We use a keyword-only argument (*) to prevent accidentally passing a value positionally
        self.value = value
        
    def is_leaf(self):
        """
        Helper method to determine if this node is a leaf.
        
        Returns:
            bool: True if it contains a prediction (value is not None), False otherwise.
        """
        return self.value is not None

In [27]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=10, n_features=None, criterion='gini', n_thresholds=None):
        """
        Args:
            min_samples_split: Minimum samples required to split a node.
            max_depth: Maximum depth of the tree.
            n_features: Number of features to consider at each split (m).
            criterion: 'gini' or 'entropy'.
            n_thresholds: Number of percentile bins to check for continuous features. 
                          If None, it exhaustively checks every unique value (slower).
        """
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.n_features = n_features
        self.criterion = criterion
        self.n_thresholds = n_thresholds
        self.root = None

    def fit(self, X, y):
        self.n_features = self.n_features if self.n_features else int(np.sqrt(X.shape[1]))
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        n_samples, n_feats = X.shape
        n_labels = len(np.unique(y))

        # Check Stopping Criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Feature Randomness
        feat_idxs = np.random.choice(n_feats, self.n_features, replace=False)
        
        # Find best split
        best_feature, best_thresh = self._best_split(X, y, feat_idxs)

        # If no valid split found
        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)

        # Split and recurse
        left_idxs, right_idxs = self._split(X[:, best_feature], best_thresh)
        left_child = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feature, threshold=best_thresh, left=left_child, right=right_child)

    def _best_split(self, X, y, feat_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        for feat_idx in feat_idxs:
            X_column = X[:, feat_idx]
            
            # --- THE THRESHOLD LOGIC SWITCH ---
            if self.n_thresholds is None:
                # Exhaustive Search: Check every unique value
                thresholds = np.unique(X_column)
            else:
                # Optimized Search: Check specific percentiles
                percentiles = np.linspace(0, 100, self.n_thresholds + 2)[1:-1]
                thresholds = np.percentile(X_column, percentiles)
            # ----------------------------------

            for thr in thresholds:
                gain = self._information_gain(y, X_column, thr)

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_threshold = thr

        return split_idx, split_threshold

    def _information_gain(self, y, X_column, threshold):
        parent_impurity = self._calculate_impurity(y)

        left_idxs, right_idxs = self._split(X_column, threshold)
        if len(left_idxs) == 0 or len(right_idxs) == 0:
            return 0
        
        n = len(y)
        n_l, n_r = len(left_idxs), len(right_idxs)
        e_l = self._calculate_impurity(y[left_idxs])
        e_r = self._calculate_impurity(y[right_idxs])
        
        child_impurity = (n_l / n) * e_l + (n_r / n) * e_r
        gain = parent_impurity - child_impurity
        
        return gain

    def _calculate_impurity(self, y):
        if self.criterion == 'gini':
            return self._gini(y)
        elif self.criterion == 'entropy':
            return self._entropy(y)
        else:
            raise ValueError("Criterion must be 'gini' or 'entropy'")

    def _gini(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        return 1.0 - np.sum(probabilities ** 2)

    def _entropy(self, y):
        counts = np.bincount(y)
        probabilities = counts / len(y)
        probabilities = probabilities[probabilities > 0]
        return -np.sum(probabilities * np.log2(probabilities))

    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs

    def _most_common_label(self, y):
        counter = Counter(y)
        return counter.most_common(1)[0][0]

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [28]:
class RandomForest:
    def __init__(self, n_trees=50, max_depth=10, min_samples_split=2, n_features=None, criterion='gini', n_thresholds=None):
        """
        Args:
            n_trees: The number of trees in the forest.
            max_depth: Maximum depth of each individual tree.
            min_samples_split: Minimum samples required to split a node.
            n_features: Number of features to consider at each split.
            criterion: 'gini' or 'entropy'.
            n_thresholds: Optimization parameter. Number of percentiles to evaluate per split. 
                          Pass None for exhaustive (np.unique) search.
        """
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_features = n_features
        self.criterion = criterion
        self.n_thresholds = n_thresholds
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        print(f"Training a Random Forest with {self.n_trees} trees...")
        
        for i in range(self.n_trees):
            X_sample, y_sample = self._bootstrap_samples(X, y)
            
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                n_features=self.n_features,
                criterion=self.criterion,
                n_thresholds=self.n_thresholds # Passed down to the tree
            )
            
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)
            
            if (i + 1) % 10 == 0 or (i + 1) == self.n_trees:
                print(f"  [{i + 1}/{self.n_trees}] trees grown.")

    def _bootstrap_samples(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        predictions = np.array([self._majority_vote(preds) for preds in tree_preds])
        return predictions

    def _majority_vote(self, y_preds):
        counter = Counter(y_preds)
        return counter.most_common(1)[0][0]

In [31]:
import numpy as np
import itertools
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- 1. LOAD THE DATA ---
print("Loading data...")
X_train = np.load('X_train_final.npz')['data']
y_train = np.load('y_train.npz')['data']
X_test = np.load('X_test_final.npz')['data']
y_test = np.load('y_test.npz')['data']

# --- 2. ENCODE LABELS TO INTEGERS ---
print("Encoding labels...")
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# --- 3. DEFINE THE HYPERPARAMETER GRID ---
param_grid = {
    'n_trees': [20, 40],               # Testing a small vs medium ensemble
    'max_depth': [8, 15],              # Testing shallow vs deep trees
    'min_samples_split': [2, 10],      # Testing high variance vs low variance splits
    'criterion': ['gini'],             # Sticking to Gini for speed
    'n_thresholds': [3, 5]             # Testing our custom optimization parameter!
}

keys, values = zip(*param_grid.items())
hyperparameter_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"\nTotal combinations to test: {len(hyperparameter_combinations)}")
print("="*50)

# --- 4. TRACK THE BEST MODEL ---
best_accuracy = 0.0
best_params = None
best_predictions = None
best_metrics = {}

# --- 5. THE GRID SEARCH LOOP ---
for i, params in enumerate(hyperparameter_combinations):
    print(f"\n--- Testing Model {i+1}/{len(hyperparameter_combinations)} ---")
    print(f"Parameters: {params}")
    
    # Initialize the customized Random Forest
    rf = RandomForest(
        n_trees=params['n_trees'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        criterion=params['criterion'],
        n_thresholds=params['n_thresholds']
    )
    
    # Fit the model
    print("  Training...")
    # rf.fit(X_train[:50], y_train[:50])  # Using only the first 50 samples for training to speed up the grid search
    rf.fit(X_train, y_train)  # Uncomment this line to use the full training set (will be slower)
    
    # Predict
    print("  Predicting...")
    y_pred = rf.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"  Results -> Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
    
    # Update best model
    if acc > best_accuracy:
        print("  🌟 New Best Model Found!")
        best_accuracy = acc
        best_params = params
        best_predictions = y_pred
        best_metrics = {'accuracy': acc, 'precision': prec, 'recall': rec}

# --- 6. PRINT AND SAVE THE FINAL RESULTS ---
print("\n" + "="*50)
print("GRID SEARCH COMPLETE")
print("="*50)
print(f"Best Accuracy: {best_accuracy:.4f}")
print("Best Parameters:")
print(json.dumps(best_params, indent=4))

# Decode predictions back to original string labels ('p0', 'p1', etc.)
y_pred_original_labels = le.inverse_transform(best_predictions)

# Save predictions for assignment submission
predictions_filename = "results_name_id.csv"
np.savetxt(predictions_filename, y_pred_original_labels, delimiter=",", fmt='%s', header="Predicted_Label", comments="")
print(f"\nSaved best predictions to: {predictions_filename}")

# Save hyperparameters for your report
with open("best_hyperparameters.json", "w") as f:
    json.dump({"best_parameters": best_params, "metrics": best_metrics}, f, indent=4)
print("Saved best hyperparameters to: best_hyperparameters.json")

Loading data...
Encoding labels...
Training data shape: (14379, 300)
Testing data shape: (1625, 300)

Total combinations to test: 16

--- Testing Model 1/16 ---
Parameters: {'n_trees': 20, 'max_depth': 8, 'min_samples_split': 2, 'criterion': 'gini', 'n_thresholds': 3}
  Training...
Training a Random Forest with 20 trees...
  [10/20] trees grown.
  [20/20] trees grown.
  Predicting...
  Results -> Accuracy: 0.6572 | Precision: 0.6620 | Recall: 0.6572
  🌟 New Best Model Found!

--- Testing Model 2/16 ---
Parameters: {'n_trees': 20, 'max_depth': 8, 'min_samples_split': 2, 'criterion': 'gini', 'n_thresholds': 5}
  Training...
Training a Random Forest with 20 trees...
  [10/20] trees grown.
  [20/20] trees grown.
  Predicting...
  Results -> Accuracy: 0.6185 | Precision: 0.6247 | Recall: 0.6185

--- Testing Model 3/16 ---
Parameters: {'n_trees': 20, 'max_depth': 8, 'min_samples_split': 10, 'criterion': 'gini', 'n_thresholds': 3}
  Training...
Training a Random Forest with 20 trees...
  [10/

In [33]:
import numpy as np
import itertools
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score

# --- 1. LOAD THE DATA ---
print("Loading data...")
X_train = np.load('X_train_final.npz')['data']
y_train = np.load('y_train.npz')['data']
X_test = np.load('X_test_final.npz')['data']
y_test = np.load('y_test.npz')['data']

# --- 2. ENCODE LABELS TO INTEGERS ---
print("Encoding labels...")
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

print(f"Training data shape: {X_train.shape}")
print(f"Testing data shape: {X_test.shape}")

# --- 3. DEFINE THE HYPERPARAMETER GRID ---
param_grid = {
    'n_trees': [80],               # Testing a medium vs large ensemble
    'max_depth': [16],              # Testing shallow vs deep trees
    'min_samples_split': [2, 10],      # Testing high variance vs low variance splits
    'criterion': ['gini'],             # Sticking to Gini for speed
}

keys, values = zip(*param_grid.items())
hyperparameter_combinations = [dict(zip(keys, v)) for v in itertools.product(*values)]

print(f"\nTotal combinations to test: {len(hyperparameter_combinations)}")
print("="*50)

# --- 4. TRACK THE BEST MODEL ---
best_accuracy = 0.0
best_params = None
best_predictions = None
best_metrics = {}

# --- 5. THE GRID SEARCH LOOP ---
for i, params in enumerate(hyperparameter_combinations):
    print(f"\n--- Testing Model {i+1}/{len(hyperparameter_combinations)} ---")
    print(f"Parameters: {params}")
    
    # Initialize the customized Random Forest
    rf = RandomForest(
        n_trees=params['n_trees'],
        max_depth=params['max_depth'],
        min_samples_split=params['min_samples_split'],
        criterion=params['criterion'],
        n_thresholds=3  # Using our custom optimization parameter for all models
    )    
    # Fit the model
    print("  Training...")
    # rf.fit(X_train[:50], y_train[:50])  # Using only the first 50 samples for training to speed up the grid search
    rf.fit(X_train, y_train)  # Uncomment this line to use the full training set (will be slower)
    
    # Predict
    print("  Predicting...")
    y_pred = rf.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f"  Results -> Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
    
    # Update best model
    if acc > best_accuracy:
        print("  🌟 New Best Model Found!")
        best_accuracy = acc
        best_params = params
        best_predictions = y_pred
        best_metrics = {'accuracy': acc, 'precision': prec, 'recall': rec}

# --- 6. PRINT AND SAVE THE FINAL RESULTS ---
print("\n" + "="*50)
print("GRID SEARCH COMPLETE")
print("="*50)
print(f"Best Accuracy: {best_accuracy:.4f}")
print("Best Parameters:")
print(json.dumps(best_params, indent=4))

# Decode predictions back to original string labels ('p0', 'p1', etc.)
y_pred_original_labels = le.inverse_transform(best_predictions)

# Save predictions for assignment submission
predictions_filename = "results_name_id.csv"
np.savetxt(predictions_filename, y_pred_original_labels, delimiter=",", fmt='%s', header="Predicted_Label", comments="")
print(f"\nSaved best predictions to: {predictions_filename}")

# Save hyperparameters for your report
with open("best_hyperparameters.json", "w") as f:
    json.dump({"best_parameters": best_params, "metrics": best_metrics}, f, indent=4)
print("Saved best hyperparameters to: best_hyperparameters.json")

Loading data...
Encoding labels...
Training data shape: (14379, 300)
Testing data shape: (1625, 300)

Total combinations to test: 2

--- Testing Model 1/2 ---
Parameters: {'n_trees': 80, 'max_depth': 16, 'min_samples_split': 2, 'criterion': 'gini'}
  Training...
Training a Random Forest with 80 trees...
  [10/80] trees grown.
  [20/80] trees grown.
  [30/80] trees grown.
  [40/80] trees grown.
  [50/80] trees grown.
  [60/80] trees grown.
  [70/80] trees grown.
  [80/80] trees grown.
  Predicting...
  Results -> Accuracy: 0.8849 | Precision: 0.8862 | Recall: 0.8849
  🌟 New Best Model Found!

--- Testing Model 2/2 ---
Parameters: {'n_trees': 80, 'max_depth': 16, 'min_samples_split': 10, 'criterion': 'gini'}
  Training...
Training a Random Forest with 80 trees...
  [10/80] trees grown.
  [20/80] trees grown.
  [30/80] trees grown.
  [40/80] trees grown.
  [50/80] trees grown.
  [60/80] trees grown.
  [70/80] trees grown.
  [80/80] trees grown.
  Predicting...
  Results -> Accuracy: 0.880